# Generate and inspect synthetic data
Run top to bottom. This notebook saves readable JSON with the original five context fields and eligible-option rewards. Conditional means are saved separately for evaluation.

In [ ]:
import os, sys, json, copy
from pathlib import Path
for key in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):
    os.environ[key]='1'
os.environ.setdefault('MPLCONFIGDIR',str(Path.cwd()/'.cache/matplotlib'))
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents]
          if (p/'contextual_bandits/simulator.py').exists())
sys.path.insert(0,str(ROOT))
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from contextual_bandits.simulator import SCENARIOS, FEATURE_NAMES, from_records
from contextual_bandits.interface import ContextPolicy
from contextual_bandits.workbench import (
    default_settings, make_world, dataset_path, world_dates, event_table, run_experiment,
    summary_table, paired_table, save_experiment, load_experiment,
    plot_world, plot_rewards, plot_regret, plot_choices, plot_shift_response)
plt.rcParams.update({'figure.dpi':110,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
defaults=default_settings()
print('Python:',sys.version.split()[0])
print('Configuration loaded. All runs save their own data and settings.')
print('Scenarios:',', '.join(SCENARIOS))

In [ ]:
SCENARIO='recurring'
SEED=1234
WORLD_ID=0
HORIZON=18250
START_DATE='2023-01-01'
world=make_world(SCENARIO,SEED,WORLD_ID,HORIZON,start_date=START_DATE)
path=dataset_path(world)
print('Saved:',path)
display(FileLink(str(path.relative_to(ROOT))))


## Inspect records, encoding and scheduled events
The full potential reward dictionary is simulator information. A deployed learner receives only the reward of its chosen action.

In [ ]:
records=world.records()
print(json.dumps(records[0],indent=2))
display(pd.DataFrame(world.contexts).head())
display(pd.DataFrame(world.x,columns=FEATURE_NAMES).head())
display(event_table(world,world_dates(world)))
birth=world.births[0][0]
print('Eligible options before / after arrival:',list(records[birth-1]['rewards']),list(records[birth]['rewards']))

In [ ]:
fig=plot_world(world,window=250)
plt.show()
assert from_records(json.loads(path.read_text()),world.mu).digest()==world.digest()
print('Saved JSON round trip verified.')